# 과제 10 · 08 · 초안을 형식으로 받고 인용을 검사하기

## 현업 시나리오
근거가 모이면 LLM 이 원인 후보와 점검 절차 초안을 씁니다. 그런데 LLM 은 근거 목록에 없는 절 번호를 지어내거나("MC01-MM 4.2.5"), 정비 기술자가 확인하기 전인데 원인을 "확정"이라고 쓰거나, 아예 형식을 깨뜨릴 수 있습니다. 이런 초안이 사람에게 그대로 가면 안 됩니다.

## 학습 목표
- `프롬프트 | 모델 | PydanticOutputParser` 로 초안을 형식 있는 객체로 받는다.
- 근거 목록에 없는 인용과 단정 표현을 코드로 찾는다.
- 형식이 깨진 출력을 검증 실패와 같게 다룬다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · 프롬프트 | 모델 | 파서

In [ ]:
# 초안 형식을 Pydantic 으로 정합니다. 근거 ID 가 하나도 없는 후보·단계는 형식부터 받지 않습니다.
# 파서가 형식 안내문(format_instructions)을 만들어 프롬프트에 넣습니다.
# 관찰 포인트: 세 조각이 | 로 이어져 하나의 Runnable 이 됩니다. 모델은 대본 JSON 을 내는 대역입니다.
import json
from langchain_core.language_models import GenericFakeChatModel
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class PracticeCause(BaseModel):
    rank: int = Field(ge=1)
    failure_type: str
    rationale: str
    evidence_ids: list[str] = Field(min_length=1)

class PracticeStep(BaseModel):
    order: int = Field(ge=1)
    instruction: str
    evidence_ids: list[str] = Field(min_length=1)

class PracticeDraft(BaseModel):
    cause_candidates: list[PracticeCause]
    inspection_steps: list[PracticeStep]

practice_parser = PydanticOutputParser(pydantic_object=PracticeDraft)
practice_prompt = ChatPromptTemplate.from_messages([
    ("system", "근거 목록에 있는 ID 만 인용하세요. 원인은 추정 후보입니다.\n{format_instructions}"),
    ("human", "근거 목록: {evidence}"),
]).partial(format_instructions=practice_parser.get_format_instructions())

practice_good_json = json.dumps({
    "cause_candidates": [{"rank": 1, "failure_type": "HDF", "rationale": "흡기 필터 막힘 가능성. 같은 유형 최근 수리 MR-0102 참고",
                          "evidence_ids": ["MC01-MM 4.2.2", "MR-0102"]}],
    "inspection_steps": [{"order": 1, "instruction": "4.2.3 의 점검 절차를 따른다", "evidence_ids": ["MC01-MM 4.2.3"]}],
}, ensure_ascii=False)

def practice_chain_with(reply):
    return practice_prompt | GenericFakeChatModel(messages=iter([reply])) | practice_parser

practice_chain = practice_chain_with(practice_good_json)
practice_evidence = ["MC01-MM 4.2.2", "MC01-MM 4.2.3", "MR-0102"]
practice_draft = practice_chain.invoke({"evidence": ", ".join(practice_evidence)})
print("체인:", type(practice_chain).__name__)
print("받은 초안:", type(practice_draft).__name__, "· 후보", [c.failure_type for c in practice_draft.cause_candidates],
      "· 단계", len(practice_draft.inspection_steps))

**결과 해설** — 체인은 `RunnableSequence` 하나이고, 모델의 JSON 글자는 파서를 지나 `PracticeDraft` 객체가 되었습니다. 이제부터 우리가 쓰는 것은 형식이 맞는 필드뿐입니다.

### 2단계 · 인용과 표현을 코드로 검사

In [ ]:
# 형식이 맞아도 내용은 틀릴 수 있습니다. 인용한 ID 가 근거 목록에 있는지, 단정 표현이 있는지 봅니다.
# 단정 표현 목록은 SOP-EA-01 7.2(정비 기술자 확인 전에는 원인을 단정하지 않는다)에서 왔습니다.
# 관찰 포인트: 지어낸 절 번호는 형식 검사를 통과합니다. 근거 목록과 대조해야 잡힙니다.
PRACTICE_ASSERTIVE = ("확정", "확실", "틀림없", "분명히")

def practice_validate(draft, evidence_ids):
    errors = []
    for label, items in (("원인 후보", draft.cause_candidates), ("점검 단계", draft.inspection_steps)):
        for item in items:
            unknown = [i for i in item.evidence_ids if i not in evidence_ids]
            if unknown:
                errors.append(f"{label}가 근거 목록에 없는 ID 를 인용했습니다: {unknown}")
            text = getattr(item, "rationale", None) or getattr(item, "instruction", "")
            if any(word in text for word in PRACTICE_ASSERTIVE):
                errors.append(f"{label}가 원인을 단정합니다: {text[:30]}")
    return errors

practice_invented = practice_chain_with(json.dumps({
    "cause_candidates": [{"rank": 1, "failure_type": "HDF", "rationale": "흡기 필터 막힘으로 확정",
                          "evidence_ids": ["MC01-MM 4.2.2"]}],
    "inspection_steps": [{"order": 1, "instruction": "필터를 교체한다", "evidence_ids": ["MC01-MM 4.2.5"]}],
}, ensure_ascii=False)).invoke({"evidence": ", ".join(practice_evidence)})
print("좋은 초안 :", practice_validate(practice_draft, practice_evidence))
for practice_error in practice_validate(practice_invented, practice_evidence):
    print("지어낸 초안:", practice_error)

**결과 해설** — 좋은 초안은 오류가 없습니다. 지어낸 초안은 형식은 통과했지만 두 가지가 잡혔습니다. 원인 후보의 "확정"(단정 표현)과, 점검 단계가 인용한 4.2.5(근거 목록에 없는 절)입니다. 이 오류 목록이 그대로 모델에게 다시 쓰라는 피드백이 됩니다.

### 3단계 · 형식이 깨진 출력

In [ ]:
# 모델이 JSON 대신 문장으로 답한 경우와, 근거 ID 를 비운 경우입니다.
# 둘 다 파서가 OutputParserException 을 냅니다. 앱은 이것을 "검사 실패"와 같게 다룹니다.
# 관찰 포인트: 형식 오류를 예외로 흘려보내면 요청 전체가 500 으로 끝납니다. 오류 목록에 넣어 다시 쓰게 합니다.
from langchain_core.exceptions import OutputParserException

practice_broken = {}
for practice_label, practice_reply in (
        ("문장으로 답함", "원인은 아마 흡기 필터 막힘입니다."),
        ("근거를 비움", json.dumps({"cause_candidates": [{"rank": 1, "failure_type": "HDF", "rationale": "필터", "evidence_ids": []}],
                                    "inspection_steps": []}))):
    try:
        practice_chain_with(practice_reply).invoke({"evidence": ", ".join(practice_evidence)})
        practice_broken[practice_label] = None
    except OutputParserException:
        practice_broken[practice_label] = "초안 형식이 맞지 않습니다"
print(practice_broken)

assert type(practice_chain).__name__ == "RunnableSequence"
assert practice_validate(practice_draft, practice_evidence) == []
practice_invented_errors = practice_validate(practice_invented, practice_evidence)
assert any("근거 목록에 없는" in e for e in practice_invented_errors), "지어낸 절 번호를 놓쳤습니다"
assert any("단정" in e for e in practice_invented_errors), "단정 표현을 놓쳤습니다"
assert all(practice_broken.values()), "형식이 깨진 출력이 통과했습니다"

**결과 해설** — 문장으로 답한 출력과 근거 ID 를 비운 출력 모두 파서에서 막혔습니다. 근거 ID 가 빈 원인 후보는 `min_length=1` 때문에 형식부터 성립하지 않습니다. **형식 검사(파서)와 내용 검사(근거 대조)는 다른 층**이고, 둘 다 있어야 합니다.

## 직접 해 볼 과제
`practice_try_reply` 의 rationale 을 "필터 막힘이 분명히 원인" 이나 "필터 막힘 가능성" 으로, evidence_ids 를 `["MR-0999"]` 로 바꿔 보세요. 어느 검사가 어떤 문장으로 잡습니까? 단정 표현 목록만으로 막을 수 없는 표현도 생각해 보세요.

In [ ]:
# 모델 출력만 바꿔 같은 체인과 검사를 돌립니다.
# 형식이 깨지면 예외, 형식이 맞으면 검사 오류 목록이 나옵니다.
practice_try_reply = json.dumps({
    "cause_candidates": [{"rank": 1, "failure_type": "HDF", "rationale": "필터 막힘이 분명히 원인", "evidence_ids": ["MR-0999"]}],
    "inspection_steps": [{"order": 1, "instruction": "4.2.3 을 따른다", "evidence_ids": ["MC01-MM 4.2.3"]}],
}, ensure_ascii=False)
print(practice_validate(practice_chain_with(practice_try_reply).invoke({"evidence": ""}), practice_evidence))

## 중간 결과
체인의 타입, 파서가 돌려준 객체, 좋은 초안과 지어낸 초안의 검사 결과, 그리고 형식이 깨진 두 출력이 파서에서 막히는 것을 확인합니다.

## 실패 경계
검사를 통과했다고 초안이 맞는 것은 아닙니다. 실제 live 실측에서 gpt-4.1-mini 의 초안 10건이 모두 검사를 통과했지만, 한 건(S02)은 출력 과다(PWF 상한) 카드에 출력 부족 갈래의 점검 절차를 섞었습니다. 인용한 절은 근거 목록에 있었기 때문입니다. 그래서 초안은 사람이 검토하기 전에는 보고서가 되지 않습니다.

## 실제 app 연결
과제 10 App 의 `drafting.py::write_draft` 가 `PROMPT | model` 다음에 `drafting.py::PARSER`(PydanticOutputParser)로 초안을 받고, 형식 오류는 `DraftParseError` 로 바꿔 검사 실패로 돌립니다. 내용 검사는 `routing.py::validate_response` 가 맡고, 단정 표현 목록은 `routing.py::ASSERTIVE` 입니다. 검사는 이것 말고도 후보 순위·단계 번호가 1부터 이어지는지, 부품 번호가 근거에 있는지를 봅니다.

## 다음 Notebook 연결
다음 `09_route_and_retry.ipynb` 에서는 경로를 규칙으로 나누고, 검사에 실패한 초안을 횟수 제한 안에서 다시 쓰게 합니다.